# 🛡️ ROAD-SHIELD: Master Deep Training Pipeline (Google Colab GPU)

**SIH 2026 Problem Statement SIH26124 | Bharat Electronics Limited (BEL) | Team 42**

This notebook trains and verifies **all 9 deep learning models** in ROAD-SHIELD on GPU:
1. **ASTM D6433 Pavement Condition Index (PCI 0-100)**: Real human polygon deduct curves (R² ≥ 0.85)
2. **HDM-4 Pavement Deterioration Forecaster**: 180-day monsoon area & depth expansion (R² ≥ 0.80)
3. **IRC:SP:83 Photometric Depth Estimator**: Cavity volume & metric depth regressor (R² ≥ 0.50)
4. **100 Hz IMU Shock Vibration Classifier**: Deep 1-D CNN on tri-axial accelerometer windows
5. **Dense Pixel Defect Segmenter**: ResNet-34 U-Net (with 100+ connected component extraction)
6. **Deep Vision Classifier**: MobileNetV3-Large / EfficientNet fine-tuning (Macro-F1 ≥ 0.75)
7. **Flagship DAN-DAG Network**: Position & Channel Attention (PAM/CAM) + 21-Node Decision DAG
8. **CNN Embedder Feature Extractor**: Mirror averaging baseline
9. **Out-of-Distribution (OOD) Input Guard**: Energy score + Outlier Exposure (AUROC ≥ 0.90)
10. **Geometric Road Marking & Zebra Detector**: Multi-crossing spatial clustering (3 zebras + 100 holes)

### Quick Start:
1. In menu: **Runtime → Change runtime type → T4 GPU**
2. Run **Cell 1** (Mount Drive & Clone Repo) and **Cell 2** (Install Dependencies)
3. Run **Cell 3: One-Click Master Pipeline Execution** to deeply train, verify, and package all models!

---
## Cell 1: Mount Google Drive & Clone Repository

In [ ]:
# Mount Google Drive for persistent checkpoints storage
from google.colab import drive
drive.mount('/content/drive')

# Clone latest repository
!git clone https://github.com/udbhav968-creator/SIH_PROJECT.git /content/SIH_PROJECT 2>/dev/null || \
    (cd /content/SIH_PROJECT && git fetch origin master && git reset --hard origin/master)

%cd /content/SIH_PROJECT
!echo "Repository ready at $(pwd)"

---
## Cell 2: Install Dependencies & Verify GPU Hardware

In [ ]:
# Install pinned dependencies
!pip install -q "scikit-learn>=1.8.0,<1.9" "opencv-python-headless>=4.8,<5" \
    onnxruntime-gpu onnx onnxscript scikit-image joblib pillow scipy timm

# Verify environment & GPU
import torch, sklearn, cv2, platform, os
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | cv2 {cv2.__version__}")
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[NOTICE] No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU")

os.environ['PYTHONPATH'] = '/content/SIH_PROJECT'
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

---
## Cell 3: 🚀 ONE-CLICK MASTER END-TO-END DEEP TRAINING PIPELINE
Runs all 12 stages in sequence, validates all quality gates, executes the 100-hole + 3-zebra stress test, and packages checkpoints to Google Drive.

In [ ]:
%cd /content/SIH_PROJECT
!python scripts/colab_master_deep_training.py --drive-dir "/content/drive/MyDrive/ROAD_SHIELD_Checkpoints"

---
## Cell 4: Fast Smoke Verification (Optional, < 3 min)
Runs a smoke test of all stages to verify weights, ONNX export, and MLOps gates without running 40 epochs.

In [ ]:
%cd /content/SIH_PROJECT
!python scripts/colab_master_deep_training.py --fast

---
## Cell 5: Multi-Defect (100 Holes) & Multi-Zebra (3 Crossings) Stress Test

In [ ]:
%cd /content/SIH_PROJECT
!python -m unittest tests.test_multi_defect_and_multi_zebra_verification

---
## Cell 6: MLOps Quality Gate Check (Automated CI Promotion Audit)

In [ ]:
%cd /content/SIH_PROJECT
!python -m mlops.ci_check

---
## Cell 7: Inspect Trained Checkpoints & Download Archive

In [ ]:
%cd /content/SIH_PROJECT
import os
print('\nTrained Models & Reports in checkpoints/:')
print('-' * 70)
for f in sorted(os.listdir('checkpoints')):
    fp = os.path.join('checkpoints', f)
    if os.path.isfile(fp):
        sz = os.path.getsize(fp) / 1e6
        print(f'  {f:<48} : {sz:7.2f} MB')
print('-' * 70)
from google.colab import files
if os.path.exists('road_shield_trained_checkpoints.zip'):
    print('Ready to download checkpoint bundle:')
    files.download('road_shield_trained_checkpoints.zip')